# Semana 11 · Sesión 2: Integración continua, issues y revisión del proyecto

**Módulo 3**

## Objetivos de la sesión

1. Leer y escribir un **workflow mínimo de GitHub Actions** que corre pytest en
   cada `push`.
2. Organizar lo pendiente con **issues** y **milestones**, y cerrar un issue
   desde un commit.
3. **Revisar el PR preliminar** del proyecto intermedio de un compañero y
   convertir la revisión que recibes en issues.

## Retomamos

En la sesión 1 escribimos pruebas y las corrimos a mano, con
`python -m pytest tests -v`. Funciona mientras alguien se acuerde de correrlas.
Hoy hacemos que las corra **GitHub**, solo, en cada `push`: eso se llama
**integración continua** (*continuous integration*, CI).

La sesión tiene dos mitades. La primera es CI, con el repositorio de tu
proyecto intermedio abierto. La segunda es el flujo de trabajo en equipo:
issues, milestones y la revisión del PR preliminar que abriste antes de esta
clase. Ten a la mano el link de ese PR.

In [ ]:
import subprocess
import sys
from pathlib import Path

## Qué es la integración continua

Cada vez que haces `push`, GitHub prende una máquina **nueva**, sin nada
instalado, y sigue una receta que tú escribes: descarga tu repositorio,
instala Python y tus dependencias y corre las pruebas. El resultado aparece en
tu Pull Request como un check.

![Flujo de integración continua: git push, una máquina nueva de GitHub Actions que instala todo y corre pytest, y un check verde o una cruz roja en el PR](img/flujo-ci.svg)

Que la máquina sea nueva es la gracia: si tu proyecto solo funciona en tu
laptop porque ahí instalaste algo a mano, o porque un archivo nunca se subió,
la CI lo descubre antes que quien revisa.

## ¿Cómo decide si pasó? El código de salida

Todo programa, al terminar, le entrega al sistema un número: su **código de
salida**. `0` quiere decir "todo bien"; cualquier otro, que algo falló. pytest
regresa `0` solo si **todas** las pruebas pasaron. GitHub no lee el reporte:
mira ese número.

Lo vemos con las pruebas de la sesión 1, que viven en esta misma carpeta. En
`main` el error plantado de `osciladores/energia.py` sigue ahí:

In [ ]:
resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q"],   # -q: reporte corto
    capture_output=True,
    text=True,
)

print(resultado.stdout.splitlines()[-1])   # la última línea: el conteo
print("Código de salida:", resultado.returncode)

Mientras el código de salida no sea `0`, el check del PR sale con una cruz
roja. Si en la sesión 1 corregiste `energia.py` en tu copia, aquí verás `0`.

## Un workflow, línea por línea

La receta es un archivo **YAML** (un formato de texto para configuraciones:
`clave: valor`, con la sangría marcando qué va dentro de qué). Junto a este
notebook está [`pruebas.yml`](pruebas.yml), un workflow mínimo:

In [ ]:
print(Path("pruebas.yml").read_text())

| Línea | Qué hace |
|---|---|
| `name: Pruebas` | el nombre que verás en la pestaña **Actions** y en el check del PR |
| `on: push` / `pull_request` | **cuándo** corre: en cada push a cualquier rama y en cada PR |
| `jobs:` → `pruebas:` | un **trabajo**, con el nombre que quieras; un workflow puede tener varios |
| `runs-on: ubuntu-latest` | la máquina: un Linux nuevo cada vez |
| `steps:` | los pasos, en orden; si uno falla, los siguientes ya no corren |
| `uses: actions/checkout@v4` | un paso ya hecho por alguien más (una *action*): descarga tu repo |
| `uses: actions/setup-python@v5` | otra action: instala la versión de Python de `with:` |
| `run: pip install ...` | un comando de terminal, como los que escribes tú |
| `run: python -m pytest tests -v` | el paso que decide el check, por su código de salida |

Dos detalles que muerden:

- GitHub solo corre los workflows que están en **`.github/workflows/`**, en la
  **raíz** del repositorio. Este `pruebas.yml` no corre en ningún lado; en tu
  proyecto va en `.github/workflows/pruebas.yml`.
- En YAML la sangría **es** la estructura, y solo se vale con espacios, nunca
  con tabuladores. Un espacio de más o de menos cambia el significado.

## El check de tus tareas, por dentro

Desde la semana 3 tus PRs de entrega tienen un check ✅ o ❌. Es un workflow
como este: [`.github/workflows/autograding.yml`](../../.github/workflows/autograding.yml),
en la raíz del repositorio del curso. Es más largo porque primero averigua qué
tarea cambiaste, pero el paso que califica es el mismo:

In [ ]:
autograding = Path("../../.github/workflows/autograding.yml").read_text()

for linea in autograding.splitlines():
    if "uses:" in linea or "run: pip" in linea or "run: pytest" in linea:
        print(linea.strip())

`checkout`, `setup-python`, `pip install` y `pytest`: la misma receta. El
`checkout` sale dos veces porque ese workflow tiene dos trabajos: uno detecta
qué tarea cambió y el otro la califica, y cada uno corre en su propia máquina.
El `upload-artifact` del final guarda tu notebook ejecutado para que el
asistente lo lea.

## TODO en clase 1

En tu repositorio `proyecto-intermedio`, en la rama `desarrollo` (en pareja,
en tu propia rama, con PR hacia `desarrollo` como siempre):

1. Crea `tests/test_modelo.py` con **al menos dos pruebas** de tu `modelo.py`:
   un caso límite que conozcas, que una solución cumpla su ecuación, una
   cantidad que se conserva... Compara por equivalencia simbólica.
2. Comprueba en tu laptop que pasan: `python -m pytest tests -v`, desde la raíz
   del proyecto.
3. Agrega `pytest==9.1.1` a tu `requirements.txt` si no está.
4. Copia `pruebas.yml` a `.github/workflows/pruebas.yml` de tu proyecto.
5. Commit atómico por cada cosa, y `push`. Abre tu PR preliminar
   `desarrollo → main`: en uno o dos minutos aparece el check.

Pega abajo el link a la ejecución en la pestaña **Actions**. Si sale ❌, abre el
paso que falló y lee el reporte de abajo hacia arriba, como en la sesión 1.

Las pruebas y el workflow **no cambian la rúbrica** del proyecto intermedio:
son la práctica para el proyecto final, donde sí se piden.

*Link a tu ejecución de Actions aquí.*

## Issues: lo pendiente, por escrito

Un **issue** es una nota pública en tu repositorio: un error, una idea, algo
que falta. Se abre en la pestaña **Issues → New issue**, con un título que
diga qué hay que hacer y una descripción con lo necesario para hacerlo.

Cada issue tiene un número, `#3`, y GitHub lo convierte en link en cualquier
comentario, commit o PR del mismo repositorio. Desde la semana 3 te pedimos
que si algo del curso no funciona abras un issue; ahora los usas para **tu**
trabajo.

## Milestones: lo pendiente, con fecha

Un **milestone** agrupa issues con una fecha límite: **Issues → Milestones →
New milestone**. GitHub muestra cuántos issues del grupo están cerrados, como
una barra de avance. Para el proyecto intermedio: un milestone
**"Entrega final"** con la fecha de la clase de la semana 12, y dentro, un
issue por cada cosa que falta.

## Cerrar un issue desde un commit

Si el mensaje de un commit o la descripción de un PR dice `closes #3`, GitHub
cierra el issue `#3` **cuando ese cambio llega a `main`**. Antes no: un commit
en `desarrollo` solo deja el link. Para el proyecto, eso pasa al fusionar el PR
de la entrega final.

```
Agrega el límite de ángulo pequeño al periodo (closes #3)
```

La palabra va en inglés (`closes`, `fixes` o `resolves`) aunque el resto del
mensaje esté en español: es la que GitHub busca, como los nombres de la API de
SymPy.

## La revisión del PR preliminar

Tu PR preliminar `desarrollo → main` ya está abierto. Hoy un compañero lo
revisa y tú revisas el suyo, con todo lo de la semana 10: comentarios de línea
concretos (qué pasa, por qué importa, qué propones), bloques `suggestion` y un
cierre con *Comment* o *Request changes*. A la lista de cotejo de la semana 10
agrégale dos preguntas:

| Pregunta | Por qué importa |
|---|---|
| ¿Hay pruebas, y prueban algo de física? | `assert 1 == 1` pasa siempre y no comprueba nada |
| ¿Pasa el check? Si no, ¿por qué? | un ❌ suele ser un archivo sin subir o una dependencia sin fijar |

No apruebes (*Approve*) un PR preliminar: todavía no está listo para
fusionarse, y la revisión es justo para que lo esté.

## TODO en clase 2

El profesor te asigna a quién revisar.

1. Abre el PR preliminar del repositorio `proyecto-intermedio` de tu
   compañero, en **Files changed**.
2. Lee el README primero: qué sistema es y qué pregunta responde. Sin eso no
   se puede juzgar si el código la responde.
3. Deja **al menos tres comentarios de línea**: uno sobre la física, uno sobre
   la organización (módulo, notebooks, README) y uno sobre las pruebas o el
   check.
4. Cierra la revisión con **Comment** o **Request changes** y un resumen de dos
   o tres frases: qué está bien y qué es lo más importante de corregir.

Pega abajo el link a tu revisión.

*Link a tu revisión aquí.*

## TODO en clase 3

Ahora del lado del autor, con la revisión que recibiste:

1. Crea el milestone **"Entrega final"** en tu repositorio, con la fecha de la
   clase de la semana 12.
2. Convierte **al menos dos** comentarios de la revisión en issues del
   milestone. En cada comentario, el menú **⋯ → Reference in new issue** crea el
   issue con el link de vuelta; luego asígnale el milestone en la barra
   lateral.
3. Resuelve uno: un commit en `desarrollo` cuyo mensaje termine con
   `(closes #N)`, y en el hilo de la revisión contesta qué cambiaste.
4. Pega abajo el link a tu milestone.

El issue va a seguir abierto hasta que fusiones tu PR en la entrega final: ese
es el momento en que el cambio llega a `main`.

*Link a tu milestone aquí.*

## Resumen

La **integración continua** corre tus pruebas en una máquina nueva en cada
`push` y en cada PR. La receta es un workflow YAML en `.github/workflows/`:
cuándo corre (`on`), en qué máquina (`runs-on`) y qué pasos sigue (`steps`,
con `uses` para actions ya hechas y `run` para comandos). El check sale verde
solo si pytest termina con código de salida `0`.

Los **issues** dejan por escrito lo pendiente, los **milestones** lo agrupan con
fecha, y `closes #N` en un commit o PR cierra el issue cuando el cambio llega a
`main`. Una revisión recibida se vuelve una lista de issues.

**Tarea de la semana:** [`../tarea/tarea-11.ipynb`](../tarea/tarea-11.ipynb).
Escribes pruebas, y el autograding comprueba que **atrapen errores**. Las
instrucciones están en [`../tarea/README.md`](../tarea/README.md).

**Proyecto intermedio:** la entrega final es antes de la clase de la semana
12: revisión contestada hilo por hilo, PR fusionado a `main` y tag
`entrega-final` (ver [`../../semana-10/proyecto/README.md`](../../semana-10/proyecto/README.md)).

**Próxima semana — Semana 12:** empieza el Módulo 4. Abrimos SymPy por
dentro: cómo representa una expresión como árbol, cómo están organizadas sus
clases y qué métodos llama solo. Es el primer paso para crear objetos
matemáticos propios, que es el proyecto final.